# FIT3182 Assignment 2 — Task 1: MongoDB Data Model

**Student IDs:** 34080678 | 33590982

This notebook implements **Task 1** of Assignment 2, covering the design and creation of MongoDB collections for the traffic speed camera system.

- **Task 1.1** — Collection Design: Vehicle, Camera, and Violation collections (schema, sample document, indexes, shard key, retention policy)
- **Task 1.2** — Collection Relationships: embed vs. reference justification, read/write patterns, trade-offs

**Dataset overview:**

| File | Description |
|------|-------------|
| `vehicle.csv` | Registered vehicles (plate, owner, type) |
| `camera.csv` | Speed camera metadata (location, speed limit) |
| `camera_event_A/B/C.csv` | Real-time speed reading events from 3 producers (used in Task 2+) |
| `camera_event_historic.csv` | Historic speed violations (used in Task 2+) |


In [ ]:
# ── Installation & Imports ──────────────────────────────────────────────────
# pymongo is pre-installed in this environment.
# To reinstall: python -m pip install pymongo --target <venv>/Lib/site-packages

import pymongo
from pymongo import MongoClient, ASCENDING
import pandas as pd
import json
import pathlib

# change the data directory path if your setup differs
DATA_DIR = pathlib.Path("/home/student/Assignment 2/FIT3182/data")

print(f"pymongo version: {pymongo.__version__}")
print("All required libraries imported successfully.")
print(f"DATA_DIR: {DATA_DIR}  (exists={DATA_DIR.exists()})")

## MongoDB Connection

Connect to the MongoDB instance running in Docker on port **27017**.  
The database `fit3182_db` is used throughout this assignment.

In [ ]:
# ── MongoDB Connection ───────────────────────────────────────────────────────
# Both containers are on Docker's default bridge network; name-based DNS is
# not supported there, so we use fervent_hamilton's bridge IP directly.
# get your MongoDB IP using docker inspect fervent_hamilton --format "{{.NetworkSettings.IPAddress}}" 
MONGO_HOST = "172.17.0.3"
MONGO_PORT = 27017
DB_NAME    = "fit3182_db"

client = MongoClient(MONGO_HOST, MONGO_PORT, serverSelectionTimeoutMS=5000)

# Verify connectivity
try:
    client.admin.command("ping")
    print(f"Connected to MongoDB at {MONGO_HOST}:{MONGO_PORT}")
except Exception as e:
    print(f"Connection failed: {e}")
    raise

db = client[DB_NAME]
print(f"Using database: '{DB_NAME}'")

---
## Task 1.1 — Collection Design

Three collections are designed for this system:

| Collection | Purpose |
|---|---|
| `vehicle` | Registered vehicle and owner information |
| `camera` | Speed camera locations and speed limits |
| `violation` | Detected speeding violations (written by streaming pipeline in Task 2) |

---
### Collection 1: `vehicle`

**Description:**  
Stores registered vehicle data including the owner's details and vehicle type. The `car_plate` field uniquely identifies each vehicle and acts as a natural primary key and the foreign reference used in the `violation` collection.

**Document Schema:**

```json
{
  "car_plate":         "string  — unique licence plate (PK)",
  "owner_name":        "string  — full name of registered owner",
  "owner_addr":        "string  — residential address",
  "vehicle_type":      "string  — e.g. Sedan, SUV, Coupe, Hatchback",
  "registration_date": "Date    — ISO 8601 date of registration"
}
```

**Sample Document:**
```json
{
  "car_plate": "ABC123",
  "owner_name": "Jane Smith",
  "owner_addr": "12 Example St, Melbourne VIC 3000",
  "vehicle_type": "Sedan",
  "registration_date": "2024-01-01T00:00:00Z"
}
```

**Indexes:**
| Field | Type | Reason |
|---|---|---|
| `car_plate` | Unique ascending | Primary lookup key; enforces uniqueness |
| `vehicle_type` | Ascending | Supports analytics queries filtered by vehicle type |

**Shard Key:** `car_plate` — high cardinality, evenly distributed, used in cross-collection joins.

**Retention Policy:** Records are kept indefinitely; vehicles deregistered > 7 years ago may be archived to cold storage.


In [ ]:
# ── Collection 1: vehicle ────────────────────────────────────────────────────
# Drop and recreate for a clean run
db.drop_collection("vehicle")

# Create collection with JSON Schema validation
db.create_collection("vehicle", validator={
    "$jsonSchema": {
        "bsonType": "object",
        "required": ["car_plate", "owner_name", "vehicle_type", "registration_date"],
        "properties": {
            "car_plate":         {"bsonType": "string",   "description": "Unique licence plate"},
            "owner_name":        {"bsonType": "string",   "description": "Registered owner name"},
            "owner_addr":        {"bsonType": "string",   "description": "Owner address"},
            "vehicle_type":      {"bsonType": "string",   "enum": ["Sedan","SUV","Coupe","Hatchback","Van","Truck"]},
            "registration_date": {"bsonType": "date",     "description": "ISO date of registration"},
        }
    }
})

# Create indexes
vehicle_col = db["vehicle"]
vehicle_col.create_index([("car_plate", ASCENDING)], unique=True, name="idx_car_plate_unique")
vehicle_col.create_index([("vehicle_type", ASCENDING)],            name="idx_vehicle_type")
print("Indexes created:", list(vehicle_col.index_information().keys()))

# Load data from CSV
# Note: source CSV has a typo 'vechicle_type' — corrected to 'vehicle_type' on load
df_vehicle = pd.read_csv(DATA_DIR / "vehicle.csv")
df_vehicle.rename(columns={"vechicle_type": "vehicle_type"}, inplace=True)
df_vehicle["registration_date"] = pd.to_datetime(df_vehicle["registration_date"])
# Deduplicate on car_plate, keeping first occurrence
df_vehicle.drop_duplicates(subset="car_plate", keep="first", inplace=True)

records = df_vehicle.to_dict(orient="records")
result = vehicle_col.insert_many(records)
print(f"Inserted {len(result.inserted_ids)} vehicle documents (after deduplication)")

# Display a sample document
sample = vehicle_col.find_one({}, {"_id": 0})
print("\nSample document:")
print(json.dumps({k: str(v) for k, v in sample.items()}, indent=2))

---
### Collection 2: `camera`

**Description:**  
Stores speed camera metadata including GPS coordinates, road position, and the enforced speed limit. This is a small, **read-heavy, rarely updated** reference collection. It is referenced by both `camera_event` and `violation` to avoid duplicating speed-limit data.

**Document Schema:**

```json
{
  "camera_id":    "int     — unique camera identifier (PK)",
  "latitude":     "double  — GPS latitude",
  "longitude":    "double  — GPS longitude",
  "position":     "double  — road position marker (km)",
  "speed_limit":  "int     — enforced speed limit (km/h)"
}
```

**Indexes:**
| Field | Type | Reason |
|---|---|---|
| `camera_id` | Unique ascending | Primary lookup; joins from events and violations |
| `speed_limit` | Ascending | Supports queries filtering cameras by speed zone |

**Shard Key:** `camera_id` — monotonically increasing integer; suitable for range-based sharding.

**Retention Policy:** Permanent — camera records represent physical infrastructure. Decommissioned cameras should be marked inactive rather than deleted (to preserve historical violation context).

In [ ]:
# ── Collection 2: camera ─────────────────────────────────────────────────────
db.drop_collection("camera")

db.create_collection("camera", validator={
    "$jsonSchema": {
        "bsonType": "object",
        "required": ["camera_id", "latitude", "longitude", "speed_limit"],
        "properties": {
            "camera_id":   {"bsonType": "int",    "description": "Unique camera ID"},
            "latitude":    {"bsonType": "double", "description": "GPS latitude"},
            "longitude":   {"bsonType": "double", "description": "GPS longitude"},
            "position":    {"bsonType": "double", "description": "Road position marker (km)"},
            "speed_limit": {"bsonType": "int",    "description": "Enforced speed limit (km/h)"},
        }
    }
})

camera_col = db["camera"]
camera_col.create_index([("camera_id", ASCENDING)], unique=True, name="idx_camera_id_unique")
camera_col.create_index([("speed_limit", ASCENDING)],             name="idx_speed_limit")
print("Indexes created:", list(camera_col.index_information().keys()))

# Load data from CSV
df_camera = pd.read_csv(DATA_DIR / "camera.csv")
df_camera["camera_id"]   = df_camera["camera_id"].astype(int)
df_camera["speed_limit"] = df_camera["speed_limit"].astype(int)

records = df_camera.to_dict(orient="records")
result = camera_col.insert_many(records)
print(f"Inserted {len(result.inserted_ids)} camera documents")

sample = camera_col.find_one({}, {"_id": 0})
print("\nSample document:")
print(json.dumps(sample, indent=2))

---
### Collection 3: `violation`

**Description:**  
Stores confirmed speeding violations detected by the real-time streaming pipeline (Task 2+). Each document represents one vehicle on one date and contains an embedded array of individual violation events for that day. This design minimises document count while keeping all daily violations co-located for fast per-plate per-day queries.

This collection is **intentionally left empty** at Task 1 setup — violations will be written here by the Spark Structured Streaming consumer in the next task.

**Document Schema:**

```json
{
  "car_plate": "string  — reference to vehicle.car_plate",
  "date":      "Date    — calendar date of violations (midnight UTC)",
  "violations": [
    {
      "violation_type":  "string  — 'INSTANTANEOUS' or 'AVERAGE'",
      "camera_id_start": "int     — entry camera (same as end for instantaneous)",
      "camera_id_end":   "int     — exit camera",
      "timestamp_start": "Date    — event time at entry camera",
      "timestamp_end":   "Date    — event time at exit camera",
      "speed_reading":   "double  — recorded or computed speed (km/h)"
    }
  ]
}
```

**Sample Document:**
```json
{
  "car_plate": "ABC123",
  "date": "2024-01-01T00:00:00Z",
  "violations": [
    {
      "violation_type": "INSTANTANEOUS",
      "camera_id_start": 1,
      "camera_id_end": 1,
      "timestamp_start": "2024-01-01T08:00:04Z",
      "timestamp_end": "2024-01-01T08:00:04Z",
      "speed_reading": 125.0
    },
    {
      "violation_type": "AVERAGE",
      "camera_id_start": 1,
      "camera_id_end": 2,
      "timestamp_start": "2024-01-01T08:00:00Z",
      "timestamp_end": "2024-01-01T08:00:33Z",
      "speed_reading": 118.2
    }
  ]
}
```

**Indexes:**
| Field | Type | Reason |
|---|---|---|
| `car_plate, date` | Compound ascending (unique) | Primary upsert key; one document per vehicle per day |
| `date` | Ascending | Time-range queries across all vehicles |

**Shard Key:** `car_plate` — consistent with `vehicle` collection; co-locates a vehicle's documents across shards.

**Retention Policy:** Violations are legally significant records. Retain indefinitely. A TTL index on an `archived_at` field (e.g. 7 years) could be applied for regulatory purging if required.


In [ ]:
# ── Collection 3: violation ──────────────────────────────────────────────────
db.drop_collection("violation")

db.create_collection("violation", validator={
    "$jsonSchema": {
        "bsonType": "object",
        "required": ["car_plate", "date", "violations"],
        "properties": {
            "car_plate":  {"bsonType": "string", "description": "Ref: vehicle.car_plate"},
            "date":       {"bsonType": "date",   "description": "Calendar date of violations (midnight UTC)"},
            "violations": {
                "bsonType": "array",
                "description": "Embedded array of violation events for this vehicle/day",
                "items": {
                    "bsonType": "object",
                    "required": ["violation_type", "camera_id_start", "camera_id_end",
                                 "timestamp_start", "timestamp_end", "speed_reading"],
                    "properties": {
                        "violation_type":  {"bsonType": "string", "enum": ["INSTANTANEOUS", "AVERAGE"]},
                        "camera_id_start": {"bsonType": "int"},
                        "camera_id_end":   {"bsonType": "int"},
                        "timestamp_start": {"bsonType": "date"},
                        "timestamp_end":   {"bsonType": "date"},
                        "speed_reading":   {"bsonType": "double"},
                    }
                }
            }
        }
    }
})

violation_col = db["violation"]
# Compound unique index — one document per vehicle per day; supports upsert in Task 2
violation_col.create_index(
    [("car_plate", ASCENDING), ("date", ASCENDING)],
    unique=True, name="idx_plate_date_unique"
)
violation_col.create_index([("date", ASCENDING)], name="idx_date")
print("Indexes created:", list(violation_col.index_information().keys()))

# Collection is intentionally empty at this stage.
# Violations will be written by the Spark Structured Streaming pipeline in Task 2.
print(f"\nviolation collection created with {violation_col.count_documents({})} documents (empty — ready for Task 2)")


---
## Task 1.2 — Collection Relationships

### Relationship Diagram

```
vehicle (car_plate PK)
    │
    └──── referenced by ──── violation.car_plate

camera (camera_id PK)
    │
    ├──── referenced by ──── violation.violations[].camera_id_start
    └──── referenced by ──── violation.violations[].camera_id_end
```

---

### Design Decision: Reference vs Embed

---

#### `violation` → `vehicle` (via `car_plate`)

**Approach: Reference**

- Violations are **written once and read many times** (reports, dashboards, enforcement).
- Owner details can change (address update, ownership transfer). Referencing ensures queries always return current owner information without backfilling violation documents.
- The trade-off is a two-collection lookup (`violation` → `vehicle`), but this is acceptable since violation queries are typically filtered first by `car_plate` or `date`, and the vehicle lookup is a single-key point query on an indexed field.

---

#### `violation` → `camera` (via `camera_id_start` / `camera_id_end`)

**Approach: Reference (with daily violations embedded inside `violation`)**

- There are only 3 cameras — the `camera` collection is tiny and fits entirely in memory.
- Referencing avoids duplicating speed-limit values across every violation sub-document.
- Camera metadata (location, speed limit) is stable reference data; a join at query time is negligible.

---

#### Violations embedded within a daily `violation` document

**Approach: Embed**

- Each `violation` document groups all events for a given `car_plate` + `date` pair. Individual violation events are **embedded** as an array rather than stored as separate top-level documents.
- This pattern minimises document count and aligns with the streaming upsert pattern: the Spark consumer will `$push` new events into the existing array using a single `update_one(..., upsert=True)` call per batch, avoiding expensive per-event document creation.
- Trade-off: if a vehicle accumulates many violations in a single day the array could grow large, but in practice daily violations per vehicle are rare enough that this is not a concern.

---

### Consistency Considerations

MongoDB does not enforce foreign-key constraints. Application-level consistency is maintained by:

1. **Insert order:** `vehicle` and `camera` documents are loaded before any `violation` documents are written by the streaming pipeline.
2. **Validation schemas** on each collection reject documents with missing required fields.
3. **Unique compound index** on `(car_plate, date)` prevents duplicate daily documents from stream retries (idempotent upserts).

---

### Summary Table

| Relationship | Strategy | Reason |
|---|---|---|
| `violation.car_plate` → `vehicle` | Reference | Owner data changes; reference ensures freshness |
| `violation.violations[].camera_id_*` → `camera` | Reference | Camera list is tiny static reference data |
| Individual violation events inside `violation` | Embed | Groups daily events for efficient upsert; low growth risk |


In [ ]:
# ── Database Summary Verification ────────────────────────────────────────────
print(f"Collections in '{DB_NAME}':")
print("-" * 45)
for col_name in sorted(db.list_collection_names()):
    col = db[col_name]
    count = col.count_documents({})
    indexes = list(col.index_information().keys())
    print(f"  {col_name:<18} {count:>6} documents   indexes: {indexes}")

print("\nTask 1 setup complete.")

---
# Task 2 — Streaming Application

**Tasks covered in this section:**

| Sub-task | Marks | Description |
|---|---|---|
| 2.1.2 | 6 | Spark Structured Streaming — stream reading, join logic, state management |
| 2.1.3 | 2 | MongoDB sink integration — upsert writes, retry handling, idempotency |
| 2.1.4 | 2 | Violation detection — instantaneous & average-speed, daily record merging |

### Architecture Overview

```
camera-events-A  ──►┐
                     │  Spark Structured Streaming
camera-events-B  ──►├──► Parse JSON ──► Instantaneous violations ──►┐
                     │                                                 ├──► union ──► foreachBatch ──► MongoDB (fit3182_db.violation)
camera-events-C  ──►┘   Join A-B / B-C ──► Average-speed violations ──►┘
```

**Prerequisites:** Run all three Kafka producers (`producer_a`, `producer_b`, `producer_c`) **before** starting the streaming query below.

In [ ]:

# ── Task 2 Setup ─────────────────────────────────────────────────────────────
# Configure Kafka packages for Spark before creating the SparkSession.
# Same package versions as the Week 11 lab to ensure compatibility.
import os

os.environ['PYSPARK_SUBMIT_ARGS'] = (
    '--packages '
    'org.apache.spark:spark-streaming-kafka-0-10_2.12:3.3.0,'
    'org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0 '
    'pyspark-shell'
)

# Core PySpark imports
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType, BooleanType,
)
from pyspark.sql.functions import (
    col, from_json, to_timestamp, udf,
    unix_timestamp, lit, expr,
)

# MongoDB and utility imports
import json
import pathlib
import pandas as pd
from datetime import datetime, timezone
from pymongo import MongoClient
from pymongo.errors import PyMongoError

print("All Task 2 imports loaded.")


In [ ]:

# ── SparkSession ─────────────────────────────────────────────────────────────
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("AWAS Traffic Streaming")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark version : {spark.version}")

# ── Connection & Topic Configuration ─────────────────────────────────────────
HOST_IP    = "host.docker.internal"   # Kafka reachable from inside Docker via host
KAFKA_PORT = 9092
MONGO_HOST = "172.17.0.3"            # MongoDB container bridge IP
MONGO_PORT = 27017
DB_NAME    = "fit3182_db"

TOPICS = {
    "A": "camera-events-A",
    "B": "camera-events-B",
    "C": "camera-events-C",
}

# Streaming state parameters (documented in the assumptions table below)
WATERMARK_DELAY = "10 minutes"   # max late-arrival tolerance
JOIN_WINDOW     = "5 minutes"    # max valid camera-to-camera travel time

print(f"Kafka bootstrap  : {HOST_IP}:{KAFKA_PORT}")
print(f"MongoDB          : {MONGO_HOST}:{MONGO_PORT}  db={DB_NAME}")
print(f"Topics           : {list(TOPICS.values())}")
print(f"Watermark delay  : {WATERMARK_DELAY}")
print(f"Join window      : {JOIN_WINDOW}")

In [ ]:

# ── Load camera reference data ────────────────────────────────────────────────
# Speed limits and road positions are loaded from camera.csv so nothing
# is hardcoded in the streaming logic.
DATA_DIR = pathlib.Path("/home/student/Assignment 2/FIT3182/data")

df_camera = pd.read_csv(DATA_DIR / "camera.csv")
df_camera["camera_id"]   = df_camera["camera_id"].astype(int)
df_camera["speed_limit"] = df_camera["speed_limit"].astype(int)

# Dict: camera_id → enforced speed limit (km/h)
speed_limits = dict(zip(df_camera["camera_id"], df_camera["speed_limit"]))

# Dict: camera_id → road position marker (km)
positions = dict(zip(df_camera["camera_id"], df_camera["position"]))

def camera_distance(cam_start: int, cam_end: int) -> float:
    """Return the road distance in km between two cameras using position markers."""
    return abs(positions[cam_end] - positions[cam_start])

print("Camera speed limits :", speed_limits)
print("Camera positions    :", positions)
print(f"Distance cam1 → cam2: {camera_distance(1, 2):.2f} km")
print(f"Distance cam2 → cam3: {camera_distance(2, 3):.2f} km")

---
## Task 2.1.2 — Streaming Join Logic

### Step 1: Read Kafka Streams

Each Kafka topic is consumed as a Spark Structured Streaming DataFrame using the same pattern from the Week 11 lab:

1. `readStream.format("kafka")` → raw byte stream
2. `CAST(value AS STRING)` → JSON string
3. `from_json(...)` → typed struct
4. `to_timestamp("timestamp")` → `event_time` column used for watermarking

A **10-minute watermark** is applied immediately on each stream to bound the streaming state and allow Spark to discard expired records.

In [ ]:

# ── Event schema ─────────────────────────────────────────────────────────────
# Matches the payload produced by all three producers:
#   event_id, batch_id, car_plate, camera_id, timestamp (ISO string),
#   speed_reading, producer_id
event_schema = StructType([
    StructField("event_id",      StringType()),
    StructField("batch_id",      IntegerType()),
    StructField("car_plate",     StringType()),
    StructField("camera_id",     IntegerType()),
    StructField("timestamp",     StringType()),   # ISO 8601 string → parsed below
    StructField("speed_reading", DoubleType()),
    StructField("producer_id",   StringType()),
])


def read_kafka_stream(topic: str):
    """
    Read a Kafka topic as a Spark Structured Streaming DataFrame.

    Follows the Week 11 lab pattern:
      readStream.format('kafka') → CAST(value AS STRING) → from_json → event_time

    Args:
        topic: Kafka topic name.

    Returns:
        Streaming DataFrame with typed columns and an event_time TimestampType column.
    """
    raw = (
        spark.readStream
        .format("kafka")
        .option("kafka.bootstrap.servers", f"{HOST_IP}:{KAFKA_PORT}")
        .option("subscribe", topic)
        .option("startingOffsets", "latest")   # consume only new events from producers
        .load()
    )
    return (
        raw
        .selectExpr("CAST(value AS STRING) AS json_value")
        .select(from_json(col("json_value"), event_schema).alias("data"))
        .select("data.*")
        .withColumn("event_time", to_timestamp(col("timestamp")))
    )


# ── Create one watermarked stream per producer topic ─────────────────────────
# Watermark bounds state retention; events older than (max_event_time - WATERMARK_DELAY)
# are considered late and will not participate in future joins.
wm_a = read_kafka_stream(TOPICS["A"]).withWatermark("event_time", WATERMARK_DELAY)
wm_b = read_kafka_stream(TOPICS["B"]).withWatermark("event_time", WATERMARK_DELAY)
wm_c = read_kafka_stream(TOPICS["C"]).withWatermark("event_time", WATERMARK_DELAY)

print("Watermarked streams created:")
for label, topic in TOPICS.items():
    print(f"  wm_{label.lower()}  ←  topic '{topic}'")

---
### Step 2 — Task 2.1.4: Instantaneous Violation Detection

A vehicle is flagged for an **instantaneous violation** when its recorded `speed_reading` exceeds the speed limit of the camera that recorded the event.

**Detection logic:**  
`speed_reading > speed_limits[camera_id]`

This is evaluated independently on each of the three streams before any joining occurs. The resulting records are projected into a **unified violation schema** shared with average-speed violations, enabling a single `union` and a single MongoDB sink downstream.

In [ ]:

# ── Instantaneous violation detection ────────────────────────────────────────
# The speed_limits dict is defined outside Spark; capture it for use inside UDFs.
_speed_limits = speed_limits


@udf(BooleanType())
def exceeds_speed_limit(camera_id, speed_reading):
    """
    Return True when speed_reading exceeds the enforced limit of the recording camera.

    Args:
        camera_id    : Integer camera identifier.
        speed_reading: Recorded vehicle speed (km/h).

    Returns:
        Boolean — True = instantaneous violation.
    """
    limit = _speed_limits.get(camera_id)
    if limit is None:
        return False
    return float(speed_reading) > float(limit)


def detect_instantaneous(stream):
    """
    Filter a single camera stream for instantaneous speed violations and project
    the result into the shared violation schema used by all downstream sinks.

    Shared violation schema:
      car_plate, camera_id_start, camera_id_end,
      timestamp_start, timestamp_end,
      speed_reading, avg_speed, violation_type
    """
    return (
        stream
        .filter(exceeds_speed_limit(col("camera_id"), col("speed_reading")))
        .select(
            col("car_plate"),
            col("camera_id").alias("camera_id_start"),
            col("camera_id").alias("camera_id_end"),     # same camera for instantaneous
            col("event_time").alias("timestamp_start"),
            col("event_time").alias("timestamp_end"),
            col("speed_reading"),
            col("speed_reading").alias("avg_speed"),     # speed_reading == avg_speed here
            lit("INSTANTANEOUS").alias("violation_type"),
        )
    )


instant_a = detect_instantaneous(wm_a)
instant_b = detect_instantaneous(wm_b)
instant_c = detect_instantaneous(wm_c)

# Union violations detected across all three camera streams
instant_violations = instant_a.union(instant_b).union(instant_c)

print("Instantaneous violation streams defined for cameras A, B, C.")

---
### Step 3 — Task 2.1.2 / 2.1.4: Average-Speed Violation Detection via Stream Joins

An **average-speed violation** is detected when a vehicle's computed speed across a road segment exceeds the ending camera's speed limit.

$$\text{avg\_speed} = \frac{\text{distance (km)}}{\text{travel\_time (h)}}$$

**Join strategy (inner join with time-bounded condition):**

| Property | Value | Reason |
|---|---|---|
| Join key | `car_plate` | Match the same vehicle across two streams |
| Direction | `A.event_time < B.event_time` | B must be observed *after* A |
| Window bound | `B.event_time ≤ A.event_time + 5 min` | Discards implausibly slow/stopped vehicles and bounds state |
| Join type | Inner join | Only matched pairs are evaluated; unmatched events expire with watermark |

**Segment A → B:** cameras 1 and 2, distance = 1.0 km, ending limit = 110 km/h  
**Segment B → C:** cameras 2 and 3, distance = 1.0 km, ending limit = 90 km/h

In [ ]:

# ── A-B average-speed join (Camera 1 → Camera 2) ─────────────────────────────
# Cameras 1 and 2 are 1.0 km apart; ending camera (cam 2) limit = 110 km/h.
DIST_AB  = camera_distance(1, 2)   # 1.0 km
LIMIT_B  = speed_limits[2]         # 110 km/h

joined_ab = (
    wm_a.alias("a")
    .join(
        wm_b.alias("b"),
        expr(f"""
            a.car_plate = b.car_plate AND
            a.event_time < b.event_time AND
            b.event_time <= a.event_time + interval {JOIN_WINDOW}
        """),
        "inner",
    )
    # Compute travel time in hours and derive average speed over the segment
    .withColumn(
        "travel_time_hours",
        (unix_timestamp(col("b.event_time")) - unix_timestamp(col("a.event_time"))) / lit(3600.0),
    )
    .withColumn("avg_speed", lit(DIST_AB) / col("travel_time_hours"))
)

# Keep only pairs where computed average speed exceeds ending camera limit
avg_violations_ab = (
    joined_ab
    .filter(col("avg_speed") > lit(float(LIMIT_B)))
    .select(
        col("a.car_plate").alias("car_plate"),
        col("a.camera_id").alias("camera_id_start"),
        col("b.camera_id").alias("camera_id_end"),
        col("a.event_time").alias("timestamp_start"),
        col("b.event_time").alias("timestamp_end"),
        col("b.speed_reading").alias("speed_reading"),   # speed at ending camera
        col("avg_speed"),
        lit("AVERAGE").alias("violation_type"),
    )
)

print(f"A-B average-speed violation stream defined")
print(f"  Segment distance : {DIST_AB:.2f} km")
print(f"  Ending cam limit : {LIMIT_B} km/h")
print(f"  Join window      : {JOIN_WINDOW}")

In [ ]:

# ── B-C average-speed join (Camera 2 → Camera 3) ─────────────────────────────
# Cameras 2 and 3 are 1.0 km apart; ending camera (cam 3) limit = 90 km/h.
DIST_BC  = camera_distance(2, 3)   # 1.0 km
LIMIT_C  = speed_limits[3]         # 90 km/h

joined_bc = (
    wm_b.alias("b")
    .join(
        wm_c.alias("c"),
        expr(f"""
            b.car_plate = c.car_plate AND
            b.event_time < c.event_time AND
            c.event_time <= b.event_time + interval {JOIN_WINDOW}
        """),
        "inner",
    )
    .withColumn(
        "travel_time_hours",
        (unix_timestamp(col("c.event_time")) - unix_timestamp(col("b.event_time"))) / lit(3600.0),
    )
    .withColumn("avg_speed", lit(DIST_BC) / col("travel_time_hours"))
)

avg_violations_bc = (
    joined_bc
    .filter(col("avg_speed") > lit(float(LIMIT_C)))
    .select(
        col("b.car_plate").alias("car_plate"),
        col("b.camera_id").alias("camera_id_start"),
        col("c.camera_id").alias("camera_id_end"),
        col("b.event_time").alias("timestamp_start"),
        col("c.event_time").alias("timestamp_end"),
        col("c.speed_reading").alias("speed_reading"),   # speed at ending camera
        col("avg_speed"),
        lit("AVERAGE").alias("violation_type"),
    )
)

print(f"B-C average-speed violation stream defined")
print(f"  Segment distance : {DIST_BC:.2f} km")
print(f"  Ending cam limit : {LIMIT_C} km/h")
print(f"  Join window      : {JOIN_WINDOW}")

---
## Task 2.1.3 — MongoDB Sink Integration

Violations are persisted using a `foreachBatch` sink. This approach batches MongoDB writes per micro-batch and enables `update_one(..., upsert=True)` merging — the core mechanism for Task 2.1.4's **daily record merging** requirement.

**Key design decisions:**

| Decision | Detail |
|---|---|
| `foreachBatch` over `foreach` | Allows batch-level `$push` upserts; `foreach` can only write one row at a time |
| `update_one` with upsert | Creates a new daily document if none exists, or merges into the existing one |
| `$setOnInsert` | Sets `car_plate` and `date` only on first insert, preventing overwrite on retry |
| `$push` to `violations` array | Appends each violation event without removing previous events for that day |
| Fresh `MongoClient` per batch | Avoids serialisation issues when Spark distributes work across executor processes |
| 1-retry on write error | Handles transient Docker network blips without infinite loops |

In [ ]:

# ── MongoDB foreachBatch sink ─────────────────────────────────────────────────
def write_violations_to_mongo(batch_df, batch_id):
    """
    foreachBatch sink: persist a micro-batch of detected violations to MongoDB.

    Each row is upserted into fit3182_db.violation using the compound key
    (car_plate, date).  The violation event is appended to the embedded
    violations array via $push, so multiple violations for the same
    vehicle on the same day are merged into a single daily document.

    Idempotency note: Spark may re-deliver a batch on failure/restart.
    Duplicate violation events within the same array are possible on retry.
    The compound unique index on (car_plate, date) ensures no duplicate
    top-level documents are created.

    Args:
        batch_df : Spark DataFrame for the current micro-batch.
        batch_id : Integer micro-batch identifier (for logging).
    """
    rows = batch_df.collect()
    if not rows:
        print(f"[batch {batch_id}] Empty batch — nothing to write.")
        return

    mongo_client = None
    try:
        mongo_client = MongoClient(MONGO_HOST, MONGO_PORT, serverSelectionTimeoutMS=5000)
        col = mongo_client[DB_NAME]["violation"]

        written = 0
        skipped = 0

        for row in rows:
            ts_start = row.timestamp_start
            if ts_start is None:
                skipped += 1
                continue

            # Derive calendar date (midnight UTC) — used as the daily document key
            violation_date = datetime(
                ts_start.year, ts_start.month, ts_start.day,
                tzinfo=timezone.utc,
            )

            violation_detail = {
                "violation_type":  row.violation_type,
                "camera_id_start": int(row.camera_id_start),
                "camera_id_end":   int(row.camera_id_end),
                "timestamp_start": ts_start,
                "timestamp_end":   row.timestamp_end,
                "speed_reading":   float(row.speed_reading),
                "avg_speed":       float(row.avg_speed),
            }

            # Retry once on transient PyMongo errors
            for attempt in range(2):
                try:
                    col.update_one(
                        # Filter: one daily document per vehicle
                        {"car_plate": row.car_plate, "date": violation_date},
                        {
                            # Only set top-level fields on first insert
                            "$setOnInsert": {
                                "car_plate": row.car_plate,
                                "date":      violation_date,
                            },
                            # Append violation detail to the embedded array
                            "$push": {"violations": violation_detail},
                        },
                        upsert=True,
                    )
                    written += 1
                    break   # success — no retry needed
                except PyMongoError as exc:
                    if attempt == 0:
                        continue   # retry once
                    print(f"  [batch {batch_id}] Write failed after retry — plate={row.car_plate}: {exc}")
                    skipped += 1

        print(f"[batch {batch_id}] written={written}  skipped={skipped}  total_rows={len(rows)}")

    finally:
        if mongo_client:
            mongo_client.close()


print("write_violations_to_mongo function defined.")

In [ ]:

# ── Union all violation streams and start the streaming query ─────────────────
# Merge instantaneous violations from all cameras with average-speed violations
# from both segments into a single DataFrame.  A single foreachBatch sink handles
# all writes, keeping MongoDB connection management in one place.
all_violations = (
    instant_violations
    .union(avg_violations_ab)
    .union(avg_violations_bc)
)

# Start the streaming query.
# foreachBatch is used so each micro-batch can be collected and written to
# MongoDB using upsert logic that isn't available in standard write sinks.
query = (
    all_violations
    .writeStream
    .foreachBatch(write_violations_to_mongo)
    .option("checkpointLocation", "./checkpoints/violations")
    .outputMode("append")
    .start()
)

print("Streaming query started.")
print("Ensure producers A, B, and C are running before messages will arrive.")
print("Interrupt the kernel (Kernel → Interrupt) to stop.\n")

try:
    query.awaitTermination()
except KeyboardInterrupt:
    print("\nKeyboard interrupt received. Stopping streaming query …")
finally:
    query.stop()
    print("Streaming query stopped.")

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/opt/conda/lib/python3.8/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/opt/conda/lib/python3.8/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
  File "/opt/conda/lib/python3.8/socket.py", line 669, in readinto
    return self._sock.recv_into(b)
KeyboardInterrupt
ERROR:py4j.clientserver:There was an exception while executing the Python Proxy on the Python Side.
Traceback (most recent call last):
  File "/opt/conda/lib/python3.8/site-packages/py4j/clientserver.py", line 617, in _call_proxy
    return_value = getattr(self.pool[obj_id], method)(*params)
  File "/opt/conda/lib/python3.8/site-packages/pyspark/sql/utils.py", line 272, in call
    raise e
  File "/opt/conda/lib/python3.8/site-packages/pyspark/sql/utils.py", line 269, in call
    self.func(Da


Keyboard interrupt received. Stopping streaming query …


----------------------------------------
Exception happened during processing of request from ('127.0.0.1', 44138)
Traceback (most recent call last):
  File "/opt/conda/lib/python3.8/socketserver.py", line 316, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/opt/conda/lib/python3.8/socketserver.py", line 347, in process_request
    self.finish_request(request, client_address)
  File "/opt/conda/lib/python3.8/socketserver.py", line 360, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/opt/conda/lib/python3.8/socketserver.py", line 747, in __init__
    self.handle()
  File "/opt/conda/lib/python3.8/site-packages/pyspark/accumulators.py", line 281, in handle
    poll(accum_updates)
  File "/opt/conda/lib/python3.8/site-packages/pyspark/accumulators.py", line 253, in poll
    if func():
  File "/opt/conda/lib/python3.8/site-packages/pyspark/accumulators.py", line 257, in accum_updates
    num_updates = read_int(sel

ConnectionRefusedError: [Errno 111] Connection refused

ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "/opt/conda/lib/python3.8/site-packages/py4j/clientserver.py", line 516, in send_command
    raise Py4JNetworkError("Answer from Java side is empty")
py4j.protocol.Py4JNetworkError: Answer from Java side is empty

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/opt/conda/lib/python3.8/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/opt/conda/lib/python3.8/site-packages/py4j/clientserver.py", line 539, in send_command
    raise Py4JNetworkError(
py4j.protocol.Py4JNetworkError: Error while sending or receiving


---
## Task 2.1.2 — Assumptions & Parameters

| Parameter | Value | Justification |
|---|---|---|
| Watermark delay | `10 minutes` | Producers publish in 5-second batch cycles; 10 min comfortably handles transient Kafka delivery delays and accommodates the slowest plausible vehicle crossing the 1 km segment |
| Join window (A→B, B→C) | `5 minutes` | 1 km at 12 km/h (minimum plausible speed in heavy traffic) ≈ 5 min; excludes genuinely stopped vehicles that restart well after the window |
| Starting offsets | `latest` | Producers stream live events; `latest` avoids replaying any historic data already in Kafka log |
| Join type | Inner join | Only matched camera-pair crossings are evaluated for average-speed violations |
| Distance source | `camera.csv` `position` field | Distances derived from road position markers, not hardcoded |
| MongoDB retry | 1 retry per row | Handles transient Docker network blips; does not retry indefinitely |
| Daily violation key | `car_plate + date (midnight UTC)` | Merges all violations for the same vehicle on the same calendar day into one document via `$push` upsert |

---

### Handling Unmatched / Expired / Invalid Records

| Case | Behaviour | Reason |
|---|---|---|
| Car seen at camera A but not B (within window) | State evicted after `max_event_time − watermark_delay`; no violation raised | Average-speed detection requires evidence of passage at both endpoints |
| Late event arriving within watermark tolerance | Spark retains state; join will still match if both events are within watermark | Watermark bounds state, not individual record correctness |
| `timestamp_end ≤ timestamp_start` | Prevented by join condition `a.event_time < b.event_time` | Only strictly ordered pairs are admitted, eliminating negative travel times |
| Same car in multiple A events within window | Each A event can match eligible B events independently | Each pair is a separate potential violation; all are evaluated |
| `camera_id` not found in `speed_limits` dict | UDF returns `False`; event is not flagged | Defensive default: unknown camera should not produce spurious violations |